# Experiment D on a Colab GPU: SmellNet sensor + GC-MS fusion

This notebook runs the same script as the local repository (`src/experiments/expD_smellnet.py`) on a GPU, because the LSTM and Transformer runs take hours per run on the project laptop's CPU.

**Protocol (leakage-free):** train on SmellNet days 1-4, choose learning rate / lambda / checkpoint on day 5, test once on day 6. At test time the models only ever see sensor windows. GC-MS is used during training (D1, D2) or as the fixed prototypes of all 50 classes (D1 prediction), never as the true class's vector.

Runtime → Change runtime type → **GPU**. Results are written to Google Drive after every run, so a disconnect only loses the run in progress; re-running a cell skips finished runs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Drive and unpack the code bundle
Upload `expD_bundle.zip` (made locally by `python -m src.utils.make_colab_notebook`). It contains only code, no data.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, zipfile
WORK = '/content/drive/MyDrive/coffee-enose-fusion'
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f'{WORK}/src'):
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall(WORK)
%cd {WORK}
!ls

## 2. Get Source 2: SmellNet code (pinned commit) and data (Hugging Face)
Only `base_data/` and `gcms_processed/` are downloaded, as in the local run.

In [ ]:
!test -d data/raw/s2/SmellNet-code || git clone -q https://github.com/MIT-MI/SmellNet.git data/raw/s2/SmellNet-code
!cd data/raw/s2/SmellNet-code && git checkout -q dde1eb01a859c13789d2b226398a542f0777fc42 && git rev-parse HEAD
!python -I src/utils/download_s2.py data/raw/s2/hf
!find data/raw/s2/hf/base_data -name '*.csv' | wc -l

## 3. Sweeps
Stage order matters: `lr` (D0, D1 over the LR grid, seed 42) → `lam` (D2 lambda grid at D0's LR) → `seeds` (two more seeds at the selected settings) → `atom` (Xatom GC-MS encoding, AB12).

By default this runs CNN, LSTM and Transformer (the MLP runs are done locally). Add `mlp` to `MODELS` to run all four on the GPU.

In [ ]:
MODELS = 'cnn lstm transformer'  # the MLP family is run locally on CPU
for stage in ['lr', 'lam', 'seeds']:
    !python -m src.experiments.expD_smellnet sweep --models {MODELS} --stage {stage}

In [ ]:
# AB12 (Xspec vs Xatom), Transformer only to save time
!python -m src.experiments.expD_smellnet sweep --models transformer --ws 100 --ps 25 --stage atom

## 4. Download the run files
Copy the resulting `results/runs/expD/` folder into the local repository (same path), then run `python -m src.experiments.expD_smellnet collect` locally.

In [ ]:
!cd results/runs && zip -qr /content/expD_runs.zip expD && ls -la /content/expD_runs.zip
files.download('/content/expD_runs.zip')